In [0]:
/*==============================================================================
  SCD TYPE 2 IMPLEMENTATION - CUSTOMER DIMENSION
  
  Purpose: Demonstrates Slowly Changing Dimension Type 2 pattern for tracking
           historical changes in customer data over time.
  
  Pattern: 
    - Bronze: Raw data with duplicates and multiple versions
    - Silver: Clean deduplicated data with SCD Type 2 tracking
    - Gold:  (Optional) Aggregated/business-ready data
  
  Key Concepts:
    - valid_from: When this version became active
    - valid_to:   When this version was superseded (NULL = current)
    - is_current: Boolean flag for active record (true = current version)
    
  Author: Databricks Demo
  Date:   2026-04-23
==============================================================================*/


-- ============================================================================
-- STEP 1: ENVIRONMENT SETUP
-- ============================================================================

-- Create catalog for demo
CREATE CATALOG IF NOT EXISTS lakehouse_demo;

-- Set default catalog
USE CATALOG lakehouse_demo;

-- Create schemas for medallion architecture
CREATE SCHEMA IF NOT EXISTS bronze;  -- Raw data layer
CREATE SCHEMA IF NOT EXISTS silver;  -- Cleaned/conformed data layer
CREATE SCHEMA IF NOT EXISTS gold;    -- Business-ready aggregates


-- ============================================================================
-- STEP 2: BRONZE LAYER - SIMULATE RAW DATA INGESTION
-- ============================================================================

-- Create bronze table to simulate raw data from source systems
-- This table may contain duplicates and multiple versions of the same customer
CREATE OR REPLACE TABLE bronze.customers_raw (
  customer_id  STRING,
  full_name    STRING,
  email        STRING,
  segment      STRING,      -- Customer tier: "premium", "standard", "basic"
  country      STRING,
  _source_file STRING,      -- Metadata: which file/batch this came from
  _ingested_at TIMESTAMP    -- Metadata: when record arrived in lakehouse
) USING DELTA;

-- Initial data load (Batch 1) - Simulates first extract from CRM system
-- Note: C003 intentionally duplicated to demonstrate deduplication
INSERT INTO bronze.customers_raw VALUES
  ('C001', 'John Doe',         'john.doe@example.com',         'premium',  'USA',    'customers_20230101.csv', '2024-01-01'),
  ('C002', 'Ana Guitierrez',   'ana.guitierrez@example.com',   'standard', 'Mexico', 'customers_20230101.csv', '2024-01-01'),
  ('C003', 'Li Wei',           'li.wei@example.com',           'basic',    'China',  'customers_20230101.csv', '2024-01-01'),
  ('C003', 'Li Wei',           'li.wei@example.com',           'basic',    'China',  'customers_20230101.csv', '2024-01-01'), -- Duplicate
  ('C004', 'Emily Smith',      'emily.smith@example.com',      'premium',  'UK',     'customers_20230101.csv', '2024-01-01'),
  ('C005', 'Raj Patel',        'raj.patel@example.com',        'standard', 'India',  'customers_20230101.csv', '2024-01-01'),
  ('C006', 'Fatima Al-Farsi',  'fatima.alfarsi@example.com',  'basic',    'UAE',    'customers_20230101.csv', '2024-01-01');


-- ============================================================================
-- STEP 3: BRONZE TO SILVER - DEDUPLICATION LOGIC
-- ============================================================================

-- Create temporary view to deduplicate bronze data
-- Uses ROW_NUMBER() to keep only the latest version per unique business key
CREATE OR REPLACE TEMP VIEW silver_incoming AS
SELECT
    customer_id,
    full_name,
    email,
    segment,
    country,
    _source_file,
    CAST(_ingested_at AS DATE) AS _ingested_at,
    current_timestamp()        AS _processed_at
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id, full_name, email, segment, country
            ORDER BY _ingested_at DESC  -- Keep most recent if true duplicates exist
        ) AS _rn
    FROM bronze.customers_raw
)
WHERE _rn = 1;  -- Keep only rank 1 (most recent)


-- ============================================================================
-- STEP 4: SILVER LAYER - CREATE SCD TYPE 2 TABLE
-- ============================================================================

-- Create silver table with SCD Type 2 columns
CREATE OR REPLACE TABLE lakehouse_demo.silver.customers (
    -- Business columns
    customer_id   STRING,
    full_name     STRING,
    email         STRING,
    segment       STRING,
    country       STRING,
    
    -- Source metadata
    _source_file  STRING,
    _ingested_at  DATE,
    _processed_at TIMESTAMP,
    
    -- SCD Type 2 tracking columns
    valid_from    TIMESTAMP,    -- When this version became active
    valid_to      TIMESTAMP,    -- When superseded (NULL = current)
    is_current    BOOLEAN,      -- Flag for active version
    _updated_at   TIMESTAMP     -- When this row was last modified
) USING DELTA;


-- ============================================================================
-- STEP 5: INITIAL LOAD TO SILVER
-- ============================================================================

-- Load deduplicated data into silver with SCD Type 2 columns initialized
INSERT INTO TABLE lakehouse_demo.silver.customers 
SELECT 
    -- Business columns
    customer_id,
    full_name,
    email,
    segment,
    country,
    
    -- Metadata
    _source_file,
    _ingested_at,
    _processed_at,
    
    -- Initialize SCD Type 2 columns
    _processed_at AS valid_from,        -- Start validity from processing time
    NULL          AS valid_to,          -- NULL means current/active
    true          AS is_current,        -- Mark as current version
    current_timestamp() AS _updated_at  -- Record creation timestamp
FROM silver_incoming;


-- ============================================================================
-- STEP 6: SIMULATE NEW DATA BATCH ARRIVAL
-- ============================================================================

-- Batch 2: New data arrives with updates for C001 and C003
-- C001: Name and segment changed (John Doe → Acme Corp, premium → enterprise)
-- C003: Email changed (li.wei@example.com → new@example.com)
INSERT INTO bronze.customers_raw
  (customer_id, full_name, email, segment, country, _ingested_at)
VALUES
  ('C001', 'Acme Corp', 'acme@example.com', 'enterprise', 'ES', '2024-01-11'),
  ('C003', 'NewCo',     'new@example.com',  'standard',   'IT', '2024-01-11');


-- ============================================================================
-- STEP 7: SCD TYPE 2 UPDATE PROCESS
-- ============================================================================

-- --------------------------------------------------------------------------
-- Step 7a: EXPIRE OLD VERSIONS
-- --------------------------------------------------------------------------
-- Find all current rows that match incoming customer_ids and mark them as expired

MERGE INTO silver.customers AS target
USING (
    -- Deduplicate source to ensure 1 row per customer_id
    -- This prevents "multiple source rows matched same target" errors
    SELECT 
      customer_id,
      full_name,
      email,
      segment,
      country,
      _source_file,
      _ingested_at
    FROM (
      SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id 
            ORDER BY _ingested_at DESC  -- Keep most recent per customer
        ) AS rn
      FROM bronze.customers_raw
      WHERE _ingested_at >= '2024-01-11'  -- Filter to new batch only
    )
    WHERE rn = 1
) AS source
  ON target.customer_id = source.customer_id
  AND target.is_current = true  -- Only match current version

-- Mark matched records as expired
WHEN MATCHED THEN UPDATE SET
  target.is_current  = false,                   -- No longer current
  target._updated_at = current_timestamp(),     -- Record update time
  target.valid_to    = source._ingested_at;     -- Set end validity date


-- --------------------------------------------------------------------------
-- Step 7b: INSERT NEW VERSIONS
-- --------------------------------------------------------------------------
-- Insert the new versions of changed customers as current records

INSERT INTO silver.customers
  (customer_id, full_name, email, segment, country, _source_file, _ingested_at, _processed_at,
   valid_from, valid_to, is_current, _updated_at)
SELECT
    source.customer_id,
    source.full_name,
    source.email,
    source.segment,
    source.country,
    source._source_file,
    CAST(source._ingested_at AS DATE) AS _ingested_at,
    current_timestamp()                AS _processed_at,
    source._ingested_at                AS valid_from,  -- Validity starts from batch date
    NULL                               AS valid_to,    -- NULL = current
    true                               AS is_current,  -- Mark as active version
    current_timestamp()                AS _updated_at
FROM (
    -- Use same deduplication logic as MERGE for consistency
    SELECT 
      customer_id,
      full_name,
      email,
      segment,
      country,
      _source_file,
      _ingested_at
    FROM (
      SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id 
            ORDER BY _ingested_at DESC
        ) AS rn
      FROM bronze.customers_raw
      WHERE _ingested_at >= '2024-01-11'
    )
    WHERE rn = 1
) AS source
WHERE NOT EXISTS (
    -- Idempotency check: Don't insert if already exists
    SELECT 1 
    FROM silver.customers existing
    WHERE existing.customer_id = source.customer_id
      AND existing.valid_from  = source._ingested_at
);


-- ============================================================================
-- STEP 8: VERIFICATION QUERIES
-- ============================================================================

-- --------------------------------------------------------------------------
-- Query 1: VIEW CURRENT STATE (Type 1 behavior)
-- Shows only the latest/active version of each customer
-- --------------------------------------------------------------------------
SELECT 
    customer_id, 
    full_name, 
    segment, 
    is_current 
FROM silver.customers 
WHERE is_current = true 
ORDER BY customer_id;


-- --------------------------------------------------------------------------
-- Query 2: VIEW COMPLETE HISTORY FOR C001
-- Shows all versions of customer C001 over time
-- --------------------------------------------------------------------------
SELECT 
    customer_id, 
    full_name, 
    segment, 
    valid_from, 
    valid_to, 
    is_current
FROM silver.customers
WHERE customer_id = 'C001'
ORDER BY valid_from;


-- --------------------------------------------------------------------------
-- Query 3: POINT-IN-TIME QUERY
-- What was C001's segment on 2024-01-11?
-- This demonstrates temporal querying capability of SCD Type 2
-- --------------------------------------------------------------------------
SELECT 
    customer_id,
    full_name,
    segment,
    valid_from,
    valid_to
FROM silver.customers
WHERE customer_id = 'C001'
  AND valid_from <= '2024-01-11'                  -- Record was active by this date
  AND (valid_to > '2024-01-11' OR valid_to IS NULL) -- Not yet expired
ORDER BY valid_from;


/*==============================================================================
  NOTES:
  
  1. IDEMPOTENCY: Steps 7a and 7b can be run multiple times safely due to:
     - MERGE matches only is_current=true
     - INSERT checks NOT EXISTS before adding
     
  2. DEDUPLICATION: Both MERGE and INSERT use identical ROW_NUMBER() logic
     to ensure consistent handling of source data
     
  3. TEMPORAL QUERIES: Use valid_from and valid_to to query data "as of"
     any point in time
     
  4. PERFORMANCE: Add indexes on (customer_id, is_current) and 
     (customer_id, valid_from, valid_to) for large tables
     
  5. ALTERNATIVE: For high-velocity updates, consider Delta Lake's
     MERGE with CDC or Lakehouse Spark Declarative Pipelines AUTO CDC
==============================================================================*/